# Read and plot the original source

Same complete `CFU_10e5` readout and plotting options as `read_atst.ipynb`. Source ranges and condition annotations remain explicit; no ATST file is read. Matplotlib plots the source values directly.


In [ ]:
import math
import matplotlib.pyplot as plt
import pandas as pd
from openpyxl import load_workbook

readout_id = 'CFU_10e5'
rows, curves = [], []

sheet = load_workbook("read_shared_data/sources/pone.0216292.s002.xlsx", read_only=True, data_only=True)["10^5 cfu per ml"]
data = [row for row in sheet.iter_rows(min_row=3, values_only=True) if isinstance(row[0], (int, float))]
time = [row[0] for row in data]
for first in range(2, 29, 3):
    dose = sheet.cell(2, first).value.split(" pfu")[0]
    for replicate, column in enumerate(range(first, first + 3), 1):
        rows.append(dict(type="uninfected_control" if dose == "0" else "phage_exposed", pfu_ml=dose, replicate=str(replicate)))
        values = [row[column - 1] for row in data]
        values = [float(v[:-1].replace(",", ".")) if isinstance(v, str) and v.endswith("*") else v for v in values]
        curves.append(pd.Series(values, index=time))

# Local curve IDs preserve source order; different time grids align without interpolation.
layout = pd.DataFrame(rows)
layout["curve_id"] = [f"curve_{i}" for i in range(1, len(rows) + 1)]
readings = pd.concat(curves, axis=1).sort_index()
readings.columns = layout["curve_id"]
readings = readings.rename_axis("Time").reset_index()


In [ ]:
panels = [(None, layout)]
ncols = min(2, len(panels))
nrows = math.ceil(len(panels) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), squeeze=False)
palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
condition_colors = {}

for ax, (panel, panel_layout) in zip(axes.flat, panels):
    for condition, group in panel_layout.groupby('pfu_ml', sort=False):
        if condition not in condition_colors:
            condition_colors[condition] = palette[len(condition_colors) % len(palette)]
        color = condition_colors[condition]
        label = str(condition)
        values = readings[group["curve_id"]]
        for index, curve_id in enumerate(group["curve_id"]):
            ax.plot(
                readings["Time"], readings[curve_id], color=color, alpha=0.6,
                label=label if index == 0 else None,
            )
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("od600")
    ax.set_title(readout_id)
    ax.legend(frameon=False)

for ax in list(axes.flat)[len(panels):]:
    fig.delaxes(ax)
fig.tight_layout()
plt.show()


In [ ]:
panels = [(None, layout)]
ncols = min(2, len(panels))
nrows = math.ceil(len(panels) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), squeeze=False)
palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
condition_colors = {}

for ax, (panel, panel_layout) in zip(axes.flat, panels):
    for condition, group in panel_layout.groupby('pfu_ml', sort=False):
        if condition not in condition_colors:
            condition_colors[condition] = palette[len(condition_colors) % len(palette)]
        color = condition_colors[condition]
        label = str(condition)
        values = readings[group["curve_id"]]
        median = values.median(axis=1)
        minimum = values.min(axis=1)
        maximum = values.max(axis=1)
        ax.errorbar(
            readings["Time"], median,
            yerr=[median - minimum, maximum - median],
            fmt="o-", capsize=3, color=color, label=label,
        )
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("od600")
    ax.set_title(readout_id)
    ax.legend(frameon=False)

for ax in list(axes.flat)[len(panels):]:
    fig.delaxes(ax)
fig.tight_layout()
plt.show()
